# 04 - Colab Scale-Up

Self-contained notebook (no local-repo dependencies): full-scale reruns of Phases 4-6,
Slide-seqV2-scale data, and ablations against GraphST, STAGATE, Garfield, and SpatialDG.

Note: the original Phase 0 plan named GEO accession GSE129788 as the "messier, larger
real dataset" for this notebook. That accession turned out to be dissociated Drop-seq
scRNA-seq (Ximerakis et al. 2019, aging mouse brain) with no spatial coordinates, so it
cannot feed a spatial-neighbor graph. It was replaced with `squidpy.datasets.slideseqv2()`
-- a real, currently-shipping loader for Slide-seqV2 mouse hippocampus data -- which needs
no manual GEO/Drive download and has a different (sparser) platform profile than the
Visium crop/full datasets used locally.

**Local run status (RTX 4050, 6GB VRAM, Windows) as of this writing:**
- Baseline (Scanpy PCA+Leiden) and the *trained* EmbeddedMemoryLayer: done locally on
  Visium crop/full, real numbers in `outputs/logs/results_table.md`.
- GraphST (Long et al. 2023, verified current SOTA): installed and ran locally with no
  issues (`pip install GraphST pot scikit-misc`). Real numbers already in the results
  table; included here again for the Slide-seqV2 scale-up run.
- STAGATE (STAGATE_pyG): **blocked locally**, not a hypothetical concern -- `torch_sparse`
  has no prebuilt wheel for this project's torch (2.11.0+cu128; PyG's wheel index tops out
  at 2.9.1) and fails to build from source. Colab can pick an older, PyG-compatible
  torch/CUDA combo, so this is the first place to actually try it.
- Garfield: **blocked locally**, not hypothetical either -- it depends on `pysam`, which
  has never shipped a Windows wheel (manylinux/macOS only, confirmed via PyPI's file
  list). Colab runs Linux, so this should just work there.
- SpaCeNet: intentionally excluded from the ARI/silhouette table -- it infers a gene-gene
  graphical model, not spot clusters, so there's nothing to compute those metrics against.
- SpatialDG: no confirmed public pip-installable implementation as of this writing.

In [ ]:
!pip install -q scanpy squidpy anndata torch scikit-learn
!pip install -q GraphST pot scikit-misc

## 1. Load Slide-seqV2 (the larger, messier real dataset)

In [ ]:
import scanpy as sc
import squidpy as sq

adata = sq.datasets.slideseqv2()
print(adata.shape, adata.X.dtype)
sparsity = 1 - adata.X.nnz / (adata.X.shape[0] * adata.X.shape[1])
print(f"sparsity: {sparsity:.4f}")

In [ ]:
# Slide-seqV2 spots are not on a Visium hex/square grid, so build the spatial
# neighbor graph from raw coordinates (coord_type="generic") rather than
# the Visium-specific coord_type="grid" used for the crop/full datasets.
sc.pp.filter_cells(adata, min_counts=500)
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=6)

## 2. Full-scale rerun: baseline (PCA + Leiden)

In [ ]:
import time

start = time.time()
sc.pp.pca(adata, n_comps=50)
sc.pp.neighbors(adata)
sc.tl.leiden(adata, resolution=1.0)
baseline_time_s = time.time() - start
print(f"baseline wall time: {baseline_time_s:.2f}s, n_clusters={adata.obs['leiden'].nunique()}")

## 3. Full-scale rerun: TRAINED EmbeddedMemoryLayer

The untrained forward pass (Phase 0) only sanity-checked shapes/VRAM -- its output
was random, since nothing pushed `memory_keys`/`memory_values` away from init. This
trains it with a reconstruction loss (decoder back to PCA-50 space) plus a spatial
smoothness term computed on the spatial-neighbor graph, which is the mechanism this
project frames as "memory-addressing replacing message passing": neighboring spots
are pulled toward similar memory embeddings without an explicit GNN aggregation layer.

`lambda_spatial=10.0`, `epochs=300` were chosen via a sweep on the crop dataset
(see `src/models/train_memory_layer.py` docstring) -- training substantially past
300 epochs kept improving reconstruction loss while *degrading* both silhouette and
spatial coherence (overfitting reconstruction at the expense of structure), so more
epochs is not automatically better here.

Also note the feature dimension: feed PCA-50 features, not raw genes. `query_proj`
is a `feature_dim x feature_dim` linear layer, so raw ~15-18k gene dims would make
it a ~200M-parameter layer -- this pinned local VRAM to 5.9/6GB and 100% GPU util
for 10+ minutes before being caught and fixed. PCA-50 also matches the baseline's
own input representation, keeping the comparison on equal footing.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import silhouette_score


class EmbeddedMemoryLayer(nn.Module):
    def __init__(self, feature_dim, memory_slots=512, memory_dim=128):
        super().__init__()
        self.memory_keys = nn.Parameter(torch.randn(memory_slots, feature_dim) * 0.02)
        self.memory_values = nn.Parameter(torch.randn(memory_slots, memory_dim) * 0.02)
        self.query_proj = nn.Linear(feature_dim, feature_dim)

    def forward(self, x):
        queries = self.query_proj(x)
        attn_scores = torch.matmul(queries, self.memory_keys.T)
        attn_weights = F.softmax(attn_scores, dim=-1)
        return torch.matmul(attn_weights, self.memory_values), attn_weights


class EmbeddedMemoryAutoencoder(nn.Module):
    def __init__(self, feature_dim, memory_slots=512, memory_dim=128):
        super().__init__()
        self.memory = EmbeddedMemoryLayer(feature_dim, memory_slots, memory_dim)
        self.decoder = nn.Linear(memory_dim, feature_dim)

    def forward(self, x):
        embedding, attn_weights = self.memory(x)
        reconstruction = self.decoder(embedding)
        return reconstruction, embedding, attn_weights


def attention_entropy(attn_weights):
    eps = 1e-12
    return -(attn_weights * torch.log(attn_weights + eps)).sum(dim=-1)


def spatial_smoothness_loss(embedding, edge_index, edge_weight):
    row, col = edge_index
    diff = embedding[row] - embedding[col]
    sq_dist = (diff**2).sum(dim=-1) * edge_weight
    return sq_dist.mean()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sc.pp.pca(adata, n_comps=50)
x = torch.tensor(adata.obsm["X_pca"].copy(), dtype=torch.float32).to(device)

coo = adata.obsp["spatial_connectivities"].tocoo()
edge_index = torch.tensor([coo.row, coo.col], dtype=torch.long).to(device)
edge_weight = torch.tensor(coo.data, dtype=torch.float32).to(device)

model = EmbeddedMemoryAutoencoder(feature_dim=x.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

lambda_spatial = 10.0
epochs = 300
max_entropy = math.log(model.memory.memory_keys.shape[0])

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)
start = time.time()
for epoch in range(epochs):
    optimizer.zero_grad()
    reconstruction, embedding, attn_weights = model(x)
    recon_loss = F.mse_loss(reconstruction, x)
    spatial_loss = spatial_smoothness_loss(embedding, edge_index, edge_weight)
    loss = recon_loss + lambda_spatial * spatial_loss
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0 or epoch == epochs - 1:
        with torch.no_grad():
            med_entropy = attention_entropy(attn_weights).median().item()
        print(f"epoch {epoch:4d}  recon={recon_loss.item():.4f}  spatial={spatial_loss.item():.4f}  "
              f"entropy={med_entropy:.4f}/{max_entropy:.4f}")
elapsed = time.time() - start

peak_mb = torch.cuda.max_memory_allocated(device) / 1024**2 if device.type == "cuda" else None
print(f"memory layer training wall time: {elapsed:.4f}s, peak VRAM: {peak_mb} MB")

model.eval()
with torch.no_grad():
    _, embedding, attn_weights = model(x)
adata.obsm["X_memory_trained"] = embedding.cpu().numpy()

final_entropy = attention_entropy(attn_weights).median().item()
if final_entropy < 0.05 * max_entropy:
    print(f"WARNING: final median entropy {final_entropy:.4f} near zero -> slot collapse")

sc.pp.neighbors(adata, use_rep="X_memory_trained", key_added="memory_neighbors")
sc.tl.leiden(adata, neighbors_key="memory_neighbors", key_added="memory_cluster_trained")

mem_labels = adata.obs["memory_cluster_trained"].to_numpy()
mem_sil = silhouette_score(adata.obsm["X_memory_trained"], mem_labels)
print(f"memory layer: n_clusters={len(set(mem_labels))}, silhouette={mem_sil:.4f}")

## 4. GraphST on Slide-seqV2 (real SOTA comparison, already verified installable)

GraphST needs raw counts (it does its own HVG/normalize/log1p/scale), so this uses a
fresh copy of the raw Slide-seqV2 adata, not the one already normalized above.
`method="leiden"` avoids GraphST's default `mclust`, which needs R/rpy2.

In [ ]:
import GraphST
from GraphST.GraphST import GraphST as GraphSTModel

raw_adata = sq.datasets.slideseqv2()

start = time.time()
GraphST.preprocess(raw_adata)
GraphST.construct_interaction(raw_adata)
GraphST.add_contrastive_label(raw_adata)
GraphST.get_feature(raw_adata)

graphst_model = GraphSTModel(raw_adata, device=device, epochs=600)
raw_adata = graphst_model.train()

n_clusters_target = adata.obs["leiden"].nunique()
GraphST.clustering(raw_adata, n_clusters=n_clusters_target, method="leiden")
graphst_elapsed = time.time() - start

graphst_labels = raw_adata.obs["domain"].to_numpy()
graphst_sil = silhouette_score(raw_adata.obsm["emb"], graphst_labels)
print(f"GraphST slideseqv2: n_clusters={len(set(graphst_labels))}, "
      f"silhouette={graphst_sil:.4f}, wall_time={graphst_elapsed:.2f}s")

## 5. STAGATE (blocked locally, try here first)

Blocked locally because `torch_sparse` has no prebuilt wheel for torch 2.11.0+cu128
(PyG's wheel index tops out at 2.9.1) and fails to build from source. Colab lets you
pick an older, PyG-compatible torch/CUDA build via the runtime settings -- check
https://data.pyg.org/whl/ for the current supported combinations before running this,
and adjust the torch version installed in cell 2 accordingly if needed.

In [ ]:
import torch
print(torch.__version__)  # confirm which torch/cuda build this runtime actually has
# then match it against https://data.pyg.org/whl/ before installing:
!pip install -q torch_geometric torch_sparse torch_scatter -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install -q git+https://github.com/QIFEIDKN/STAGATE_pyG.git

## 6. Garfield (blocked locally, try here first)

Blocked locally because Garfield depends on `pysam`, which has never shipped a
Windows wheel (only manylinux/macOS on PyPI -- confirmed by checking PyPI's file
list directly, not inferred). Colab runs Linux, so `pysam` should install from a
prebuilt wheel with no issues here. Verified real PyPI name is `garfield`, not
`garfield-bio` (that name is unclaimed on PyPI as of this writing -- do not use it).

In [ ]:
!pip install -q garfield

## 7. SpatialDG

SpatialDG (Briefings in Bioinformatics 2026) has no confirmed pip-installable
package as of this writing -- check for an official implementation release
before attempting a reimplementation. Until then, its results_table.md row
stays `TODO: not yet benchmarked (Colab, if reimplementation feasible)`.